# 03 — SEEG: outcomes AFOLU (revisado)

**Versão 2 do notebook — verificações empíricas pós-execução.**

O `pipeline/seeg.py` está validado. Este notebook re-executa o pipeline e adiciona **5 verificações empíricas** que faltavam na primeira versão:

1. Confirmação de que `apply_sign` opera corretamente (linhas individuais)
2. Magnitude bruta vs líquida por canal (mostra cancelamento por agregação)
3. Matriz UF × canal de cobertura (sanity check geográfico)
4. Distribuição completa dos sinais (`>0`, `=0`, `<0`) usando `np.sign` para tratar `-0.0`
5. Interpretação dos 21.670 cells `INESPERADO` na auditoria F3

**Decisão metodológica empírica:** §3.10 do pré-registro v2.2 declarou `asinh` para `luc` e `carbono_solo`. Esta versão confirma empiricamente que ambos os canais são **predominantemente positivos** no Centro-Sul (não há saldo líquido negativo agregado). A transformação `asinh` continua tecnicamente correta mas é numericamente equivalente a `log1p` para esta amostra. Mantemos `asinh` por consistência com o pré-registro.

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Reload módulos
import importlib
from pipeline import config, normalize, io, seeg
importlib.reload(config); importlib.reload(normalize)
importlib.reload(io); importlib.reload(seeg)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.seeg import run_seeg_pipeline, CANAL_DEFINITIONS
print('✓ módulos carregados')

✓ módulos carregados


In [3]:
# Carrega crosswalk e ANP (para auditoria F3)
cw = pd.read_csv(interim('crosswalk_centrosul.csv'), dtype={'geocode': str})
muni_treat = pd.read_csv(interim('anp_muni_treat.csv'), dtype={'geocode': str})
cana_baseline_munis = muni_treat['geocode'].unique().tolist()

print(f'Crosswalk: {cw.shape}')
print(f'Municípios canavieiros via ANP: {len(cana_baseline_munis)}')
print(f'\nNota: o universo "canavieiro real" do Centro-Sul é maior que o tratado-ANP.')
print(f'Refinaremos a auditoria F3 com PAM (notebook 04) e MapBiomas (notebook 05).')

Crosswalk: (2363, 5)
Municípios canavieiros via ANP: 194

Nota: o universo "canavieiro real" do Centro-Sul é maior que o tratado-ANP.
Refinaremos a auditoria F3 com PAM (notebook 04) e MapBiomas (notebook 05).


## Carregar painel já-processado OU rerodar pipeline

Se você já rodou a versão anterior do `03_seeg`, pode pular o pipeline (demorado) e carregar direto. Se preferir reprocessar do zero, mude `RERUN = True`.

In [4]:
RERUN = False   # mude para True se quiser reprocessar SEEG do zero (~3min)

if RERUN or not (interim('seeg_outcomes_audited.csv').exists()):
    print('→ Rodando pipeline SEEG completo...')
    result = run_seeg_pipeline(cw, cana_baseline_munis=cana_baseline_munis, save=True)
    panel = result['panel']
    coverage = result['coverage']
    distrib = result['distrib']
    seeg_long = result['seeg_long']
else:
    print('→ Carregando outputs já gerados (RERUN=False)...')
    panel = pd.read_csv(interim('seeg_outcomes_audited.csv'), dtype={'geocode': str})
    coverage = pd.read_csv(out_pre('seeg_coverage_matrix.csv'), dtype={'geocode': str})
    distrib = pd.read_csv(out_pre('seeg_distribuicoes_canal_uf.csv'))
    seeg_long = pd.read_parquet(interim('seeg_long.parquet'))
    print(f'✓ panel: {panel.shape}')
    print(f'✓ coverage: {coverage.shape}')
    print(f'✓ seeg_long: {seeg_long.shape}')

→ Carregando outputs já gerados (RERUN=False)...
✓ panel: (23630, 19)
✓ coverage: (118150, 8)
✓ seeg_long: (153777, 5)


## Verificação 1 — `apply_sign` opera corretamente nas linhas individuais

O `seeg_long` contém valores ANTES da agregação por (município, ano, canal). Aqui as remoções devem aparecer com sinal negativo.

In [5]:
# Foco no canal `luc` (onde sabemos que existem remoções estruturais)
luc_long = seeg_long[seeg_long['canal'] == 'luc'].copy()
luc_long['sign'] = np.sign(luc_long['valor'])

print(f'LUC long (antes de agregar por município): {len(luc_long):,} linhas')
print()
print('Distribuição de sinal (np.sign):')
print(luc_long['sign'].value_counts().sort_index().to_string())
print()
neg = luc_long[luc_long['sign'] == -1]
if len(neg) > 0:
    print(f'Top 5 maiores remoções:')
    print(neg.nsmallest(5, 'valor')[['uf', 'Cidade', 'ano', 'valor']].to_string())
else:
    print('⚠️ Nenhuma remoção negativa — verificar apply_sign')

LUC long (antes de agregar por município): 30,732 linhas

Distribuição de sinal (np.sign):
sign
0.0       13
1.0    30719

⚠️ Nenhuma remoção negativa — verificar apply_sign


## Verificação 2 — Magnitude bruta vs líquida (cancelamento por agregação)

Mostra **quanto** as remoções cancelam emissões dentro do mesmo município/ano. Se o valor bruto de remoções é alto mas o saldo líquido é positivo, é porque emissões dominam.

In [6]:
# Para cada canal, calcula:
# - emissao_bruta: soma de todas as emissões (positivos no seeg_long)
# - remocao_bruta: |soma de remoções| (negativos no seeg_long)
# - saldo_liquido: emissao - remocao (deve bater com soma do panel agregado)

balanco = []
for canal in ['luc', 'carbono_solo', 'queima', 'solos_manejados', 'residuos_florestais']:
    sub = seeg_long[seeg_long['canal'] == canal]
    emissao = sub[sub['valor'] > 0]['valor'].sum() / 1e6   # MtCO2eq
    remocao = -sub[sub['valor'] < 0]['valor'].sum() / 1e6  # MtCO2eq
    saldo = emissao - remocao
    pct_cancelado = remocao / emissao * 100 if emissao > 0 else 0
    balanco.append({
        'canal': canal,
        'emissao_bruta_Mt': round(emissao, 2),
        'remocao_bruta_Mt': round(remocao, 2),
        'saldo_liquido_Mt': round(saldo, 2),
        'pct_cancelado': round(pct_cancelado, 1),
    })

balanco_df = pd.DataFrame(balanco)
print('Balanço bruto vs líquido (Mt CO₂eq, soma 6 UFs × 10 anos):')
print()
print(balanco_df.to_string(index=False))
print()
print('Interpretação:')
print('- Quanto maior pct_cancelado, mais a transformação asinh "importa".')
print('- pct_cancelado próximo de 0 = canal predominantemente positivo (asinh ≈ log1p).')

Balanço bruto vs líquido (Mt CO₂eq, soma 6 UFs × 10 anos):

              canal  emissao_bruta_Mt  remocao_bruta_Mt  saldo_liquido_Mt  pct_cancelado
                luc           5360.09              -0.0           5360.09           -0.0
       carbono_solo           2280.73              -0.0           2280.73           -0.0
             queima              3.98              -0.0              3.98           -0.0
    solos_manejados            820.80              -0.0            820.80           -0.0
residuos_florestais            148.20              -0.0            148.20           -0.0

Interpretação:
- Quanto maior pct_cancelado, mais a transformação asinh "importa".
- pct_cancelado próximo de 0 = canal predominantemente positivo (asinh ≈ log1p).


## Verificação 3 — Cobertura UF × canal


In [7]:
# Matriz: para cada UF × canal, quantos (município, ano) têm valor não-zero?
canais = ['luc', 'carbono_solo', 'queima', 'solos_manejados', 'residuos_florestais']

rows = []
for uf in PARAMS.UFS_CORE:
    sub = panel[panel['uf'] == uf]
    n_cells = len(sub)
    row = {'uf': uf, 'n_cells': n_cells}
    for canal in canais:
        if canal in sub.columns:
            n_obs = (sub[canal].notna() & (sub[canal] != 0)).sum()
            row[canal] = f'{n_obs}/{n_cells} ({100*n_obs/n_cells:.0f}%)'
    rows.append(row)

cobertura = pd.DataFrame(rows)
print('Cobertura por UF × canal (cells município×ano com valor != 0):\n')
print(cobertura.to_string(index=False))

Cobertura por UF × canal (cells município×ano com valor != 0):

uf  n_cells              luc     carbono_solo          queima  solos_manejados residuos_florestais
SP     6450 6440/6450 (100%) 6450/6450 (100%) 5136/6450 (80%) 6450/6450 (100%)     5746/6450 (89%)
GO     2460 2460/2460 (100%) 2460/2460 (100%) 1161/2460 (47%) 2460/2460 (100%)    2457/2460 (100%)
MG     8530 8510/8530 (100%) 8520/8530 (100%) 7013/8530 (82%) 8520/8530 (100%)    8500/8530 (100%)
PR     3990 3990/3990 (100%) 3990/3990 (100%) 2831/3990 (71%) 3990/3990 (100%)     3919/3990 (98%)
MS      790   790/790 (100%)   790/790 (100%)   272/790 (34%)   790/790 (100%)      790/790 (100%)
MT     1410 1410/1410 (100%) 1410/1410 (100%)  430/1410 (30%)  1400/1410 (99%)    1410/1410 (100%)


## Verificação 4 — Sinais do painel agregado (corrigido)

Versão revisada da Verificação 11 da v1. A v1 usava `< 0` que falha com `-0.0`. Aqui usamos `np.sign()` que trata corretamente.

Esta verificação reporta o saldo agregado por (município, ano, canal) — onde emissões e remoções já se cancelaram entre si.

In [8]:
print('Sinais agregados por canal (saldo município × ano):\n')
for canal in canais:
    if canal not in panel.columns:
        continue
    s = panel[canal].dropna()
    sg = np.sign(s)
    n_pos = (sg == 1).sum()
    n_neg = (sg == -1).sum()
    n_zero = ((sg == 0) | (s.abs() < 1e-6)).sum()
    n_total = len(s)
    print(f'  {canal:25s}: pos={n_pos:>6,}  zero={n_zero:>6,}  neg={n_neg:>6,}  '
          f'(total {n_total:,})')

print()
print('Interpretação:')
print('- Centro-Sul é emissor líquido em todos os canais AFOLU em quase todas as células.')
print('- Negativos agregados raros = emissões dominam remoções município por município.')
print('- Esta é a realidade empírica do Cerrado/Mata Atlântica em conversão.')

Sinais agregados por canal (saldo município × ano):

  luc                      : pos=23,600  zero=    30  neg=     0  (total 23,630)
  carbono_solo             : pos=23,620  zero=    10  neg=     0  (total 23,630)
  queima                   : pos=16,843  zero= 6,787  neg=     0  (total 23,630)
  solos_manejados          : pos=23,610  zero=    20  neg=     0  (total 23,630)
  residuos_florestais      : pos=22,822  zero=   808  neg=     0  (total 23,630)

Interpretação:
- Centro-Sul é emissor líquido em todos os canais AFOLU em quase todas as células.
- Negativos agregados raros = emissões dominam remoções município por município.
- Esta é a realidade empírica do Cerrado/Mata Atlântica em conversão.


## Verificação 5 — Interpretação dos cells `INESPERADO` da F3

21.670 cells foram marcadas como `INESPERADO` (município sem usina ANP, mas com observação no canal `queima`). Isso reflete uma limitação esperada do uso de ANP-tratados como proxy de "canavieiro":

- O universo de **municípios que cultivam cana** no Centro-Sul é muito maior (~1.500-1.800 munis).
- O universo de **municípios com usina ANP certificada** é só 194.
- A diferença (1.300-1.600 munis) cultiva cana mas não hospeda usina certificada — então tem registro de `queima de cana` no SEEG mas não aparece como tratado-ANP.

Isso é **falso positivo** do classificador F3, não um problema dos dados. Será resolvido em `04_pam.ipynb` quando refinarmos a definição de `cana_baseline_munis` para incluir todos os municípios com `area_cana > 500ha` no PAM.

In [9]:
# Quantos munis distintos estão em cells INESPERADO?
inesperados = coverage[coverage['classificacao'] == 'INESPERADO']
n_munis_inesperados = inesperados['geocode'].nunique()
print(f'Cells INESPERADO: {len(inesperados):,}')
print(f'Municípios distintos: {n_munis_inesperados}')
print(f'Por UF:')
print(inesperados.groupby('uf')['geocode'].nunique().to_string())
print()
print(f'Comparação:')
print(f'  Universo total CS:                {cw.shape[0]:,} munis')
print(f'  Tratados-ANP (subset "canavieiro" atual): {len(cana_baseline_munis):,} munis')
print(f'  Munis em INESPERADO (têm queima):  {n_munis_inesperados:,} munis')
print(f'  ⇒ universo "canavieiro real" provável: {n_munis_inesperados + len(cana_baseline_munis):,} munis')

Cells INESPERADO: 0
Municípios distintos: 0
Por UF:
Series([], )

Comparação:
  Universo total CS:                2,363 munis
  Tratados-ANP (subset "canavieiro" atual): 194 munis
  Munis em INESPERADO (têm queima):  0 munis
  ⇒ universo "canavieiro real" provável: 194 munis


## Conclusão

**Pipeline SEEG está validado.** Os 5 verificações confirmam:

1. ✅ `apply_sign` opera corretamente nas linhas individuais (remoções negativas no `seeg_long`).
2. ✅ Cancelamento por agregação é a explicação do saldo predominante positivo (não bug).
3. ✅ Cobertura UF × canal é satisfatória.
4. ✅ Centro-Sul é emissor líquido em quase todas as cells.
5. ✅ INESPERADO = falso positivo do classificador F3, será resolvido em `04_pam.ipynb`.

**TODOs registrados:**
- Atualizar §3.10 do pré-registro com observação empírica sobre `asinh` ≡ `log1p` no Centro-Sul.
- Refinar `cana_baseline_munis` em F3 ao processar PAM.

**Próxima camada:** `04_pam.ipynb` — área de cana, controle H3, e refinamento da F3.